# 📊 Sprint 2: Z-Score e a Régua Universal
### Minicurso: Informática Biomédica Aplicada | Jornada InfoBio 2026

---

## Você chegou ao Sprint 2 — hora de equilibrar o jogo! ⚖️

No Sprint 1 você visualizou os biomarcadores. Mas há um problema oculto: a **CK pode chegar a 800 U/L** enquanto o **Cortisol fica em torno de 15 µg/dL**. Para um algoritmo de IA, isso parece que a CK é cinquenta vezes mais importante — o que é um erro grave.

É como comparar um elefante com uma formiga na mesma balança: os números gritam tamanhos diferentes, mas ambos são igualmente importantes para o ecossistema.

**🎯 Objetivo deste Sprint:**
> Aplicar a fórmula matemática de normalização (**Z-Score**) para equilibrar as escalas dos dados — resolvendo o problema dos **Elefantes vs Formigas** — e preparar a base para o Machine Learning do Sprint 3.

---

## ⚠️ O Problema das Escalas

Imagine que você é um algoritmo de IA e recebe estes dois valores de um atleta:

- **CK (Creatina Quinase):** 572 U/L
- **Cortisol:** 13 µg/dL

Se a IA ler esses números *em bruto*, ela vai pensar: *"572 é muito maior que 13, então CK é com certeza a variável mais importante."*

**Mas isso está errado!** A CK tem essa escala grande porque é medida em U/L — uma unidade diferente. O Cortisol de 13 µg/dL pode ser perfeitamente normal, enquanto uma CK de 572 U/L também pode ser esperada após um jogo intenso.

A IA não sabe a diferença entre **magnitude de valor** e **importância clínica**. Nós precisamos ensiná-la.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Carregando os dados dos 22 atletas direto do GitHub
url = "https://raw.githubusercontent.com/FBRosito/jornada-infobio-2026/main/dados_atletas_minicurso.csv"
df = pd.read_csv(url)

# Evidenciando o problema: veja a diferença brutal de escalas
print('Estatísticas de CK e Cortisol:')
df[['CK_UL', 'Cortisol_ugdL']].describe().round(2)

## 🧮 A Solução: Z-Score (Escore Padrão)

O **Z-Score** responde à seguinte pergunta para cada atleta:

> *"Esse atleta está quantos desvios padrão acima ou abaixo da média do grupo?"*

A fórmula é:

$$Z = \frac{X - \mu}{\sigma}$$

Em linguagem humana:
- **Z = 0** → O atleta está exatamente na média do grupo
- **Z = +2** → O atleta está muito acima da média (possível alerta)
- **Z = -1** → O atleta está um pouco abaixo da média

Depois da transformação, **CK e Cortisol ficam na mesma escala** — e a IA pode comparar as duas de forma justa.

### Implementando manualmente com Pandas

Em vez de usar uma biblioteca pronta, vamos calcular o Z-Score *na mão* usando `mean()` e `std()` do Pandas. Isso deixa o processo transparente — você entende cada etapa.

In [ ]:
# Calculando Z-Score manualmente para CK e Cortisol
df['CK_zscore']      = (df['CK_UL']      - df['CK_UL'].mean())      / df['CK_UL'].std()
df['Cortisol_zscore'] = (df['Cortisol_ugdL'] - df['Cortisol_ugdL'].mean()) / df['Cortisol_ugdL'].std()

print('Estatísticas após normalização (Z-Score):')
print('CK normalizada — média:', round(df['CK_zscore'].mean(), 6), '| desvio padrão:', round(df['CK_zscore'].std(), 2))
print('Cortisol normalizado — média:', round(df['Cortisol_zscore'].mean(), 6), '| desvio padrão:', round(df['Cortisol_zscore'].std(), 2))
print()
df[['ID_Atleta', 'CK_UL', 'CK_zscore', 'Cortisol_ugdL', 'Cortisol_zscore']].head(8)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# CK original vs normalizada
sns.histplot(df['CK_UL'], ax=axes[0, 0], color='steelblue', bins=10)
axes[0, 0].set_title('CK Original (U/L)', fontsize=12)
axes[0, 0].set_xlabel('CK (U/L)')

sns.histplot(df['CK_zscore'], ax=axes[0, 1], color='steelblue', bins=10)
axes[0, 1].set_title('CK Normalizada (Z-Score)', fontsize=12)
axes[0, 1].set_xlabel('Z-Score')
axes[0, 1].axvline(0, color='red', linestyle='--', label='Média do grupo')
axes[0, 1].legend()

# Cortisol original vs normalizado
sns.histplot(df['Cortisol_ugdL'], ax=axes[1, 0], color='coral', bins=10)
axes[1, 0].set_title('Cortisol Original (µg/dL)', fontsize=12)
axes[1, 0].set_xlabel('Cortisol (µg/dL)')

sns.histplot(df['Cortisol_zscore'], ax=axes[1, 1], color='coral', bins=10)
axes[1, 1].set_title('Cortisol Normalizado (Z-Score)', fontsize=12)
axes[1, 1].set_xlabel('Z-Score')
axes[1, 1].axvline(0, color='red', linestyle='--', label='Média do grupo')
axes[1, 1].legend()

plt.suptitle('A forma da distribuição se mantém — só a escala muda', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n✅ Perceba: CK e Cortisol agora têm a MESMA escala. A IA pode comparar as duas de forma justa!')

In [ ]:
# Normalizando todos os biomarcadores para deixar o DataFrame pronto
df['LDH_zscore']          = (df['LDH_UL']          - df['LDH_UL'].mean())          / df['LDH_UL'].std()
df['Testosterona_zscore'] = (df['Testosterona_nmolL'] - df['Testosterona_nmolL'].mean()) / df['Testosterona_nmolL'].std()
df['PCR_zscore']          = (df['PCR_mgL']          - df['PCR_mgL'].mean())          / df['PCR_mgL'].std()

df_norm = df[['ID_Atleta', 'Posicao', 'CK_zscore', 'Cortisol_zscore', 'LDH_zscore', 'PCR_zscore', 'Testosterona_zscore']].copy()

print('DataFrame normalizado — todos os biomarcadores na mesma escala:')
print(f'Shape: {df_norm.shape}')
df_norm.head()

---

## 🏆 DESAFIO 2

In [ ]:
# ✏️ SEU CÓDIGO AQUI
